# D-FINE Object Detection with OpenVINO

**D-FINE** (Redefine Regression Task of DETRs as Fine-grained Distribution Refinement) is a DETR-family object detector that redefines the bounding-box regression task as fine-grained distribution refinement, improving localization accuracy over standard DETR. It is available in five sizes: `n`, `s`, `m`, `l`, `x`.

This notebook walks through the full workflow for the `n` (nano) model on the COCO validation set: export the model to OpenVINO IR and run GPU inference.


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Setup](#Setup)
- [Checkpoints](#Checkpoints)
- [XPU (PyTorch)](#XPU-(PyTorch))
- [Export to OpenVINO IR](#Export-to-OpenVINO-IR)
- [OpenVINO GPU inference](#OpenVINO-GPU-inference)
- [Precision recommendation](#Precision-recommendation)



<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/d-fine/dfine.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

```bash
# Install the required packages specific to this notebook.
sudo apt update && sudo apt install -y \
  git

# Install Intel GPU runtime for OpenCL for using Intel GPU device with OpenVINO
sudo add-apt-repository -y ppa:kobuk-team/intel-graphics
sudo apt update
sudo apt install -y --no-install-recommends \
    libze-intel-gpu1 \
    intel-opencl-icd

# Create a Python 3.13 environment. Conda can be used to manage environments:
conda create -n dfine313 python=3.13
conda activate dfine313
pip install ipykernel nbconvert
# conda deactivate
# conda env remove -n dfine313
```

**Python 3.13** is recommended to run this notebook.

In [ ]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("dfine.ipynb")


In [ ]:
from notebook_utils import device_widget

device = device_widget()

device.value


## Setup
[back to top ⬆️](#Table-of-contents:)

Run `setup.sh` (from this folder). It will:

- Clone the upstream D-FINE repo into `dfine_repo/` (pinned to commit `956d170`) — skipped if already present.
- Copy the OpenVINO-specific scripts (`ov_export.py`, `ov_infer.py`, `two_stage_topk.py`, `xpu_infer.py`) into `dfine_repo/`.
- Install XPU `torch`/`torchvision` from the Intel wheel index, plus the remaining Python requirements.

## Checkpoints
[back to top ⬆️](#Table-of-contents:)

Pretrained COCO checkpoints are expected in `dfine_repo/checkpoints/`:

```
dfine_repo/checkpoints/
              ├── dfine_n_coco.pth
              ├── dfine_s_coco.pth
              ├── dfine_m_coco.pth
              ├── dfine_l_coco.pth
              └── dfine_x_coco.pth
```

Download the checkpoint for the model being evaluated from the official D-FINE release. 

## XPU (PyTorch)
[back to top ⬆️](#Table-of-contents:)

Run from `dfine_repo/` (the script import `src.*`):

```bash
python -u xpu_infer.py --model n --batch-size 8 --num-workers 0
```

- `python -u` — run Python with unbuffered stdout so progress streams live.
- `xpu_infer.py` — PyTorch COCO-val evaluation script (runs on the XPU by default).
- `--model n` — model size (`n`/`s`/`m`/`l`/`x`); selects the config yaml and `dfine_n_coco.pth` checkpoint.
- `--batch-size 8` — validation batch size.
- `--num-workers 0` — dataloader workers; `0` keeps loading in the main process (avoids XPU dataloader issues).

`xpu_infer.py` targets Intel GPU through its default `--device xpu`: it creates `torch.device(args.device)`, moves the model, criterion, and postprocessor with `.to(device)`, and passes that device to D-FINE's existing `evaluate(...)` loop.

## Export to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Run from `dfine_repo/`:

```bash
m="n"             # model variant: n / s / m / l / x
precision="fp16"  # fp16 or auto-opt

python -u ov_export.py --model $m --precision $precision
```

- `--model $m` — model variant (e.g. `n`); loads the matching PyTorch checkpoint.
- `--precision fp16` — export with plain FP16 weights.
- `--precision auto-opt` — export with NNCF automatic mixed-precision optimization (int8/fp16 where it helps).

The exported OpenVINO IR (`.xml`/`.bin`) will be written under the output directory in `dfine_repo/checkpoints/ov`.

`ov_export.py` wraps the detector and postprocessor in deploy mode, converts them with `ov.convert_model(...)`, optionally quantizes the backbone with `nncf.quantize(...)`, rewrites TopK to keep the best eight classes per query before selecting the final detections globally, and saves the IR with `ov.save_model(...)`.

## OpenVINO GPU inference
[back to top ⬆️](#Table-of-contents:)

```bash
python -u ov_infer.py --model $m --precision $precision --device GPU --batch-size 8
```

- `--model $m` — model variant; locates the exported IR for this variant.
- `--precision <fp16|auto-opt>` — must match the precision used at export time.
- `--device GPU` — run on the GPU.
- `--batch-size 8` — validation batch size.

`ov_infer.py` creates `ov.Core()`, loads the IR with `core.read_model(...)`, fixes the requested input shape using `model.reshape(...)`, compiles it for `--device` with `core.compile_model(..., {"INFERENCE_PRECISION_HINT": "f16"})`, and calls the compiled model for each validation batch.

## Precision recommendation
[back to top ⬆️](#Table-of-contents:)

| Model | Recommended precision |
|-------|-----------------------|
| n, s  | fp16                  |
| m, l, x | auto-opt/fp16       |